# Validation architecture

This notebook runs the opt-in `delaunay` binary to generate deterministic validation-level failure examples. It renders the five-level validation architecture and keeps ordinary notebook artifacts under `target/`.

Generated files are written under `target/notebooks/01_validation/`. Set `DELAUNAY_VALIDATION_DOC_FIGURE_DIR=docs/assets/validation` when intentionally refreshing the canonical diagrams reused by documentation and papers.


## 1. Generate validation cases

The CLI produces the case geometry and diagnostics. The notebook only renders that artifact.

In [ ]:
"""Generate deterministic validation failure cases with the local delaunay binary."""

import json
import math
import os
import shutil
import subprocess
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, cast

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon


def find_repo_root(start: Path) -> Path:
    """Return the repository root containing the project marker files."""
    for path in (start, *start.parents):
        if (path / "Cargo.toml").is_file() and (path / "pyproject.toml").is_file():
            return path
    message = "Run this notebook from inside the delaunay repository."
    raise RuntimeError(message)


def positive_int_env(name: str, default: int) -> int:
    """Read a positive integer environment override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be a positive integer, got {raw_value!r}") from error
    if value <= 0:
        raise ValueError(f"{name} must be positive, got {value}")
    return value


def run_command(command: list[str], *, cwd: Path, timeout: int) -> subprocess.CompletedProcess[str]:
    """Run one repository command with captured output and a timeout."""
    print("$", " ".join(command))
    result = subprocess.run(  # noqa: S603 - command is an argv list with shell=False and controlled cwd.
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        message = f"command failed with exit code {result.returncode}: {' '.join(command)}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        raise RuntimeError(message)
    return result


def delaunay_command_prefix(root: Path) -> list[str]:
    """Return the command prefix for the local `delaunay` CLI."""
    configured = os.environ.get("DELAUNAY_BINARY")
    if configured is not None:
        if not configured.strip():
            message = "DELAUNAY_BINARY must not be empty"
            raise ValueError(message)
        binary = Path(configured).expanduser()
        binary = binary if binary.is_absolute() else root / binary
        if not binary.is_file():
            raise FileNotFoundError(f"DELAUNAY_BINARY does not point to a file: {binary}")
        return [str(binary)]

    cargo = shutil.which("cargo")
    if cargo is None:
        message = "cargo executable was not found on PATH; set DELAUNAY_BINARY to a built binary"
        raise RuntimeError(message)
    return [cargo, "run", "--profile", "perf", "--features", "cli", "--bin", "delaunay", "--"]


def tracked_figure_dir_from_env(name: str, root: Path, expected_relative: Path) -> Path | None:
    """Return the tracked paper figure directory when explicitly enabled."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return None
    if not raw_value.strip():
        message = f"{name} must not be empty"
        raise ValueError(message)
    configured = Path(raw_value)
    if configured.is_absolute():
        message = f"{name} must be the repo-relative path {expected_relative.as_posix()!r}"
        raise ValueError(message)
    figure_dir = (root / configured).resolve()
    expected = (root / expected_relative).resolve()
    if figure_dir != expected:
        message = f"{name} must be the repo-relative path {expected_relative.as_posix()!r}"
        raise ValueError(message)
    return figure_dir


ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_STEM = "01_validation"
OUTPUT_DIR = ROOT / "target" / "notebooks" / NOTEBOOK_STEM
DEMO_PATH = OUTPUT_DIR / "validation_demo.json"
VALIDATION_FIGURE_DIR = OUTPUT_DIR / "validation_levels"
DOC_FIGURE_DIR = tracked_figure_dir_from_env("DELAUNAY_VALIDATION_DOC_FIGURE_DIR", ROOT, Path("docs/assets/validation"))
TIMEOUT_SECONDS = positive_int_env("DELAUNAY_VALIDATION_DEMO_TIMEOUT_SECONDS", 120)


def save_figure_png(figure: Any, png_path: Path, *, dpi: int = 180) -> None:
    """Save a notebook PNG and an explicitly enabled canonical copy."""
    png_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(png_path, dpi=dpi, facecolor=figure.get_facecolor())
    if DOC_FIGURE_DIR is not None:
        tracked_png_path = DOC_FIGURE_DIR / png_path.name
        tracked_png_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(tracked_png_path, dpi=dpi, facecolor=figure.get_facecolor())


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    *delaunay_command_prefix(ROOT),
    "validation-demo",
    "--output",
    str(DEMO_PATH),
]

start_time = time.perf_counter()
run_command(command, cwd=ROOT, timeout=TIMEOUT_SECONDS)
elapsed = time.perf_counter() - start_time

print(f"Repository: {ROOT}")
print(f"Validation demo JSON: {DEMO_PATH}")
print(f"Generated validation demo cases in {elapsed:.2f}s")

## 2. Load and check the artifact

The parser validates the small JSON contract before plotting so the rendered table tracks the binary schema.

In [ ]:
"""Load and validate the generated validation-demo JSON artifact."""

type JsonObject = dict[str, Any]
type Point2 = tuple[float, float]


@dataclass(frozen=True, slots=True)
class ValidationPoint:
    """Parsed labeled 2D point from validation-demo visual metadata."""

    label: str
    coordinates: Point2


@dataclass(frozen=True, slots=True)
class CircumcircleWitness:
    """Parsed finite positive circumcircle witness for a Level 5 visual."""

    center: Point2
    radius: float


@dataclass(frozen=True, slots=True)
class ValidationVisual:
    """Parsed visual metadata used by plotting and independent witness checks."""

    points: tuple[ValidationPoint, ...]
    simplices: tuple[tuple[int, ...], ...]
    highlighted_simplices: frozenset[int]
    highlighted_edges: tuple[tuple[int, int], ...]
    invalid_points: frozenset[int]
    isolated_points: frozenset[int]
    duplicate_simplices: tuple[tuple[int, ...], ...]
    circumcircle: CircumcircleWitness | None


@dataclass(frozen=True, slots=True)
class ValidationCase:
    """Parsed validation-demo case consumed by the notebook figures."""

    level: int
    layer: str
    title: str
    status: str
    public_check: str
    public_reference: str
    input_summary: str
    explanation: str
    diagnostic: str
    visual: ValidationVisual


def reject_json_constant(value: str) -> object:
    """Reject non-standard JSON numeric constants before artifact parsing."""
    raise ValueError(f"JSON artifact contains non-finite value {value}")


def load_json_object(path: Path) -> JsonObject:
    """Load a JSON object from disk."""
    with path.open(encoding="utf-8") as handle:
        data = json.load(handle, parse_constant=reject_json_constant)
    if not isinstance(data, dict):
        raise TypeError(f"expected JSON object in {path}")
    return cast("JsonObject", data)


def require_object(value: Any, context: str) -> JsonObject:
    """Return a JSON object or raise a contextual type error."""
    if not isinstance(value, dict):
        raise TypeError(f"{context} must be a JSON object")
    return cast("JsonObject", value)


def require_list(value: Any, context: str) -> list[Any]:
    """Return a JSON list or raise a contextual type error."""
    if not isinstance(value, list):
        raise TypeError(f"{context} must be a list")
    return value


def require_str(value: Any, context: str) -> str:
    """Return a JSON string or raise a contextual type error."""
    if not isinstance(value, str):
        raise TypeError(f"{context} must be a string")
    return value


def require_number(value: Any, context: str) -> float:
    """Return a finite JSON number as float or raise a contextual error."""
    if isinstance(value, bool) or not isinstance(value, int | float):
        raise TypeError(f"{context} must be a finite JSON number")
    number = float(value)
    if not math.isfinite(number):
        raise ValueError(f"{context} must be finite, got {number!r}")
    return number


def require_int(value: Any, context: str) -> int:
    """Return a JSON integer or raise a contextual type error."""
    if type(value) is not int:
        raise TypeError(f"{context} must be an integer")
    return value


def point_coordinates(record: JsonObject, context: str) -> Point2:
    """Parse one 2D point coordinate array."""
    coordinates = require_list(record.get("coordinates"), f"{context}.coordinates")
    if len(coordinates) != 2:
        raise ValueError(f"{context}.coordinates must have length 2")
    return (require_number(coordinates[0], f"{context}.x"), require_number(coordinates[1], f"{context}.y"))


def parse_int_tuple(value: Any, context: str) -> tuple[int, ...]:
    """Parse a JSON integer list into immutable indices."""
    return tuple(require_int(item, f"{context}[{index}]") for index, item in enumerate(require_list(value, context)))


def parse_simplex_tuple(value: Any, context: str) -> tuple[tuple[int, ...], ...]:
    """Parse a JSON simplex list into immutable vertex-index tuples."""
    return tuple(parse_int_tuple(item, f"{context}[{index}]") for index, item in enumerate(require_list(value, context)))


def parse_edge_tuple(value: Any, context: str) -> tuple[tuple[int, int], ...]:
    """Parse a JSON edge list into immutable point-index pairs."""
    edges: list[tuple[int, int]] = []
    for index, raw_edge in enumerate(require_list(value, context)):
        edge = parse_int_tuple(raw_edge, f"{context}[{index}]")
        if len(edge) != 2:
            raise ValueError(f"{context}[{index}] must contain exactly two point indices")
        edges.append((edge[0], edge[1]))
    return tuple(edges)


def require_bounded_index(index: int, upper_bound: int, context: str) -> int:
    """Return an index after proving it addresses the parsed visual array."""
    if index < 0 or index >= upper_bound:
        raise IndexError(f"{context} index {index} is outside 0..{upper_bound - 1}")
    return index


def require_triangle_simplex(simplex: tuple[int, ...], context: str) -> tuple[int, ...]:
    """Return a 2D visual simplex after proving it has three vertices."""
    if len(simplex) != 3:
        raise ValueError(f"{context} must contain exactly three point indices")
    return simplex


def parse_circumcircle(value: Any, context: str) -> CircumcircleWitness | None:
    """Parse an optional finite positive circumcircle witness."""
    if value is None:
        return None
    circle = require_object(value, context)
    center_raw = require_list(circle.get("center"), f"{context}.center")
    if len(center_raw) != 2:
        raise ValueError(f"{context}.center must have length 2")
    center = (require_number(center_raw[0], f"{context}.center.x"), require_number(center_raw[1], f"{context}.center.y"))
    radius = require_number(circle.get("radius"), f"{context}.radius")
    if radius <= 0.0:
        raise ValueError(f"{context}.radius must be positive, got {radius}")
    return CircumcircleWitness(center=center, radius=radius)


def parse_visual_point(raw_point: Any, context: str) -> ValidationPoint:
    """Parse one labeled visual point."""
    point = require_object(raw_point, context)
    return ValidationPoint(label=require_str(point.get("label"), f"{context}.label"), coordinates=point_coordinates(point, context))


def validate_visual_point_indices(visual: ValidationVisual, context: str) -> None:
    """Verify that all parsed visual indices address parsed points or simplices."""
    for simplex_index, simplex in enumerate(visual.simplices):
        for offset, point_index in enumerate(simplex):
            require_bounded_index(point_index, len(visual.points), f"{context}.simplices[{simplex_index}][{offset}]")
    for simplex_index in sorted(visual.highlighted_simplices):
        require_bounded_index(simplex_index, len(visual.simplices), f"{context}.highlighted_simplices")
    for edge_index, edge in enumerate(visual.highlighted_edges):
        for offset, point_index in enumerate(edge):
            require_bounded_index(point_index, len(visual.points), f"{context}.highlighted_edges[{edge_index}][{offset}]")
    for offset, point_index in enumerate(visual.invalid_points):
        require_bounded_index(point_index, len(visual.points), f"{context}.invalid_points[{offset}]")
    for offset, point_index in enumerate(visual.isolated_points):
        require_bounded_index(point_index, len(visual.points), f"{context}.isolated_points[{offset}]")
    for duplicate_index, duplicate in enumerate(visual.duplicate_simplices):
        for offset, point_index in enumerate(duplicate):
            require_bounded_index(point_index, len(visual.points), f"{context}.duplicate_simplices[{duplicate_index}][{offset}]")


def parse_visual(raw_visual: Any, context: str) -> ValidationVisual:
    """Parse validation-demo visual metadata once at the JSON boundary."""
    visual = require_object(raw_visual, context)
    points = tuple(
        parse_visual_point(raw_point, f"{context}.points[{index}]") for index, raw_point in enumerate(require_list(visual.get("points"), f"{context}.points"))
    )
    simplices = tuple(
        require_triangle_simplex(simplex, f"{context}.simplices[{index}]")
        for index, simplex in enumerate(parse_simplex_tuple(visual.get("simplices"), f"{context}.simplices"))
    )
    duplicate_simplices = tuple(
        require_triangle_simplex(simplex, f"{context}.duplicate_simplices[{index}]")
        for index, simplex in enumerate(parse_simplex_tuple(visual.get("duplicate_simplices"), f"{context}.duplicate_simplices"))
    )
    parsed = ValidationVisual(
        points=points,
        simplices=simplices,
        highlighted_simplices=frozenset(parse_int_tuple(visual.get("highlighted_simplices"), f"{context}.highlighted_simplices")),
        highlighted_edges=parse_edge_tuple(visual.get("highlighted_edges"), f"{context}.highlighted_edges"),
        invalid_points=frozenset(parse_int_tuple(visual.get("invalid_points"), f"{context}.invalid_points")),
        isolated_points=frozenset(parse_int_tuple(visual.get("isolated_points"), f"{context}.isolated_points")),
        duplicate_simplices=duplicate_simplices,
        circumcircle=parse_circumcircle(visual.get("circumcircle"), f"{context}.circumcircle"),
    )
    validate_visual_point_indices(parsed, context)
    return parsed


def case_context(index: int) -> str:
    """Return the JSON path label for a validation-demo case."""
    return "valid_baseline" if index < 0 else f"cases[{index}]"


def parse_case(raw_case: Any, index: int) -> ValidationCase:
    """Parse the fields each notebook-rendered case needs."""
    context = case_context(index)
    case = require_object(raw_case, context)
    return ValidationCase(
        level=require_int(case.get("level"), f"{context}.level"),
        layer=require_str(case.get("layer"), f"{context}.layer"),
        title=require_str(case.get("title"), f"{context}.title"),
        status=require_str(case.get("status"), f"{context}.status"),
        public_check=require_str(case.get("public_check"), f"{context}.public_check"),
        public_reference=require_str(case.get("public_reference"), f"{context}.public_reference"),
        input_summary=require_str(case.get("input_summary"), f"{context}.input_summary"),
        explanation=require_str(case.get("explanation"), f"{context}.explanation"),
        diagnostic=require_str(case.get("diagnostic"), f"{context}.diagnostic"),
        visual=parse_visual(case.get("visual"), f"{context}.visual"),
    )


validation_demo = load_json_object(DEMO_PATH)
if validation_demo.get("schema") != "delaunay.validation_demo":
    raise ValueError(f"unexpected schema: {validation_demo.get('schema')!r}")
if validation_demo.get("schema_version") != 1:
    raise ValueError(f"unexpected schema version: {validation_demo.get('schema_version')!r}")
if validation_demo.get("dimension") != 2:
    raise ValueError(f"expected 2D validation demo, got {validation_demo.get('dimension')!r}")

valid_baseline = parse_case(validation_demo.get("valid_baseline"), -1)
validation_cases = [parse_case(raw_case, index) for index, raw_case in enumerate(require_list(validation_demo.get("cases"), "cases"))]
levels = [case.level for case in validation_cases]
if levels != [1, 2, 3, 4, 5]:
    raise ValueError(f"expected validation levels [1, 2, 3, 4, 5], got {levels}")

print(f"baseline: {valid_baseline.title} -> {valid_baseline.diagnostic}")
for case in validation_cases:
    print(f"Level {case.level}: {case.layer} -> {case.status}")

## 3. Prepare validation figures

The notebook writes both the validation hierarchy overview and one generated PNG per validation level. The paper owns captions and explanatory text.

In [ ]:
"""Prepare output paths for overview and per-level validation figures."""


HIERARCHY_FIGURE_PATH = OUTPUT_DIR / "validation_hierarchy.png"

VALIDATION_FIGURE_SLUGS = {
    1: "element_validity",
    2: "combinatorial_consistency",
    3: "intrinsic_pl_topology",
    4: "valid_realization",
    5: "geometric_predicates",
}
OBSOLETE_VALIDATION_FIGURE_NAMES = (
    "validation_model_failures.png",
    "validation_level_4_valid_affine_realization.png",
)


def validation_level_figure_path(case: ValidationCase) -> Path:
    """Return the generated PNG path for one validation level."""
    try:
        slug = VALIDATION_FIGURE_SLUGS[case.level]
    except KeyError as error:
        raise ValueError(f"unsupported validation level for paper figure: {case.level}") from error
    return VALIDATION_FIGURE_DIR / f"validation_level_{case.level}_{slug}.png"


def clear_stale_validation_figures() -> None:
    """Remove obsolete notebook-owned validation images before rendering fresh ones."""
    figure_dirs = [OUTPUT_DIR, VALIDATION_FIGURE_DIR]
    if DOC_FIGURE_DIR is not None:
        figure_dirs.append(DOC_FIGURE_DIR)

    for figure_dir in figure_dirs:
        for path in figure_dir.glob("validation_level_*.png"):
            path.unlink()
        for filename in OBSOLETE_VALIDATION_FIGURE_NAMES:
            path = figure_dir / filename
            if path.exists():
                path.unlink()


VALIDATION_FIGURE_PATHS = tuple(validation_level_figure_path(case) for case in validation_cases)
ALL_VALIDATION_FIGURE_PATHS = (HIERARCHY_FIGURE_PATH, *VALIDATION_FIGURE_PATHS)

print("validation figures:")
print(f"- {HIERARCHY_FIGURE_PATH}")
for figure_path in VALIDATION_FIGURE_PATHS:
    print(f"- {figure_path}")

## 4. Render validation figures

The overview diagram summarizes the layer stack; the per-level images intentionally contain only the geometric witness. Paper captions and surrounding prose explain the validation layer.

In [ ]:
"""Render generated validation cases as overview and separate visual figures."""


def visual_points(visual: ValidationVisual) -> list[tuple[str, Point2]]:
    """Return labeled 2D points from parsed visual metadata."""
    return [(point.label, point.coordinates) for point in visual.points]


def visual_simplices(visual: ValidationVisual) -> list[tuple[int, ...]]:
    """Return parsed 2D visual simplices as a mutable drawing sequence."""
    return list(visual.simplices)


def visual_circle(visual: ValidationVisual) -> tuple[Point2, float] | None:
    """Return the parsed circumcircle witness in Matplotlib-friendly form."""
    if visual.circumcircle is None:
        return None
    return (visual.circumcircle.center, visual.circumcircle.radius)


def point_by_index(points: list[Point2], index: int, context: str) -> Point2:
    """Return a point by visual index with contextual bounds checking."""
    if index < 0 or index >= len(points):
        raise IndexError(f"{context} index {index} is outside 0..{len(points) - 1}")
    return points[index]


def simplex_points(points: list[Point2], simplex: tuple[int, ...], context: str) -> list[Point2]:
    """Return the three points for a 2D simplex visual."""
    if len(simplex) != 3:
        raise ValueError(f"{context} must contain exactly three point indices")
    return [point_by_index(points, point_index, f"{context}[{offset}]") for offset, point_index in enumerate(simplex)]


VISUAL_AREA_TOLERANCE = 1.0e-12
VISUAL_CIRCLE_TOLERANCE = 1.0e-9


@dataclass(frozen=True, slots=True)
class VisualWitness:
    """Parsed visual metadata used for independent case invariant checks."""

    circumcircle: CircumcircleWitness | None
    points: list[Point2]
    simplices: list[tuple[int, ...]]
    highlighted_simplices: frozenset[int]
    invalid_points: frozenset[int]
    isolated_points: frozenset[int]
    duplicate_simplices: list[tuple[int, ...]]


@dataclass(frozen=True, slots=True)
class ValidationHierarchyLayer:
    """One row in the validation hierarchy overview diagram."""

    level: int
    name: str
    scope: tuple[str, ...]
    questions: tuple[str, ...]
    color: str


def canonical_simplex(simplex: tuple[int, ...]) -> tuple[int, ...]:
    """Return the orientation-independent simplex vertex set."""
    return tuple(sorted(simplex))


def simplex_multiplicities(simplices: list[tuple[int, ...]]) -> dict[tuple[int, ...], int]:
    """Count simplices by their abstract vertex set."""
    counts: dict[tuple[int, ...], int] = {}
    for simplex in simplices:
        key = canonical_simplex(simplex)
        counts[key] = counts.get(key, 0) + 1
    return counts


def triangle_signed_area(points: list[Point2], simplex: tuple[int, ...], context: str) -> float:
    """Return the signed area of a 2D simplex visual."""
    a, b, c = simplex_points(points, simplex, context)
    return 0.5 * ((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))


def point_distance(left: Point2, right: Point2) -> float:
    """Return Euclidean distance between two rendered points."""
    return math.hypot(left[0] - right[0], left[1] - right[1])


def circle_tolerance(radius: float) -> float:
    """Return the scale-aware tolerance for rendered circumcircle witnesses."""
    return VISUAL_CIRCLE_TOLERANCE * max(1.0, radius)


def validate_duplicate_simplex_witness(case_index: int, level: int, witness: VisualWitness) -> None:
    """Verify that duplicate-simplex visual witnesses are real duplicates."""
    if level == 2 and not witness.duplicate_simplices:
        raise ValueError(f"cases[{case_index}] Level 2 must include a duplicate simplex witness")
    counts = simplex_multiplicities(witness.simplices)
    for duplicate_index, duplicate in enumerate(witness.duplicate_simplices):
        multiplicity = counts.get(canonical_simplex(duplicate), 0)
        if multiplicity < 2:
            raise ValueError(f"cases[{case_index}].visual.duplicate_simplices[{duplicate_index}] does not duplicate an emitted simplex")


def validate_isolated_point_witness(case_index: int, level: int, witness: VisualWitness) -> None:
    """Verify that isolated-point witnesses are unused by every simplex."""
    used_points = {point_index for simplex in witness.simplices for point_index in simplex}
    for point_index in sorted(witness.isolated_points):
        if point_index in used_points:
            raise ValueError(f"cases[{case_index}].visual.isolated_points contains used point index {point_index}")
    if level == 3:
        unused_points = set(range(len(witness.points))) - used_points
        if witness.isolated_points != unused_points:
            message = f"cases[{case_index}] Level 3 isolated points {sorted(witness.isolated_points)} do not match unused points {sorted(unused_points)}"
            raise ValueError(message)


def validate_degenerate_simplex_witness(case_index: int, level: int, witness: VisualWitness) -> None:
    """Verify that Level 4 highlighted simplices are geometrically degenerate."""
    if level != 4:
        return
    if not witness.highlighted_simplices:
        raise ValueError(f"cases[{case_index}] Level 4 must highlight a degenerate simplex")
    for simplex_index in sorted(witness.highlighted_simplices):
        area = abs(triangle_signed_area(witness.points, witness.simplices[simplex_index], f"cases[{case_index}].visual.simplices[{simplex_index}]"))
        if area > VISUAL_AREA_TOLERANCE:
            raise ValueError(f"cases[{case_index}].visual.simplices[{simplex_index}] area {area} exceeds degeneracy tolerance {VISUAL_AREA_TOLERANCE}")


def validate_circumcircle_witness(case_index: int, level: int, witness: VisualWitness) -> None:
    """Verify that Level 5 circumcircle metadata witnesses a Delaunay violation."""
    if level != 5:
        return
    circle = witness.circumcircle
    if circle is None:
        raise ValueError(f"cases[{case_index}] Level 5 must include a circumcircle witness")
    if not witness.highlighted_simplices:
        raise ValueError(f"cases[{case_index}] Level 5 must highlight a circumcircle-defining simplex")
    if not witness.invalid_points:
        raise ValueError(f"cases[{case_index}] Level 5 must mark an interior invalid point")
    center = circle.center
    radius = circle.radius
    tolerance = circle_tolerance(radius)
    for simplex_index in sorted(witness.highlighted_simplices):
        for vertex_index in witness.simplices[simplex_index]:
            vertex = point_by_index(witness.points, vertex_index, f"cases[{case_index}].visual.simplices[{simplex_index}]")
            residual = abs(point_distance(vertex, center) - radius)
            if residual > tolerance:
                raise ValueError(f"cases[{case_index}] simplex vertex {vertex_index} has circumcircle residual {residual}, tolerance {tolerance}")
    for point_index in sorted(witness.invalid_points):
        point = point_by_index(witness.points, point_index, f"cases[{case_index}].visual.invalid_points")
        distance = point_distance(point, center)
        if distance >= radius - tolerance:
            message = f"cases[{case_index}] invalid point {point_index} is not strictly inside the circumcircle: distance {distance}, radius {radius}"
            raise ValueError(message)


def validate_case_visual_invariants(case: ValidationCase, case_index: int) -> None:
    """Verify that rendered visual metadata independently witnesses the claimed failure."""
    level = case.level
    visual = case.visual
    labeled_points = visual_points(visual)
    points = [point for _, point in labeled_points]
    simplices = visual_simplices(visual)
    witness = VisualWitness(
        circumcircle=visual.circumcircle,
        points=points,
        simplices=simplices,
        highlighted_simplices=visual.highlighted_simplices,
        invalid_points=visual.invalid_points,
        isolated_points=visual.isolated_points,
        duplicate_simplices=list(visual.duplicate_simplices),
    )
    if level == 1 and not witness.invalid_points:
        raise ValueError(f"cases[{case_index}] Level 1 must mark the invalid point")
    validate_duplicate_simplex_witness(case_index, level, witness)
    validate_isolated_point_witness(case_index, level, witness)
    validate_degenerate_simplex_witness(case_index, level, witness)
    validate_circumcircle_witness(case_index, level, witness)


def axes_limits(points: list[Point2], circle: tuple[Point2, float] | None) -> tuple[float, float, float, float]:
    """Return padded axis limits that include points and optional circle."""
    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    if circle is not None:
        center, radius = circle
        xs.extend([center[0] - radius, center[0] + radius])
        ys.extend([center[1] - radius, center[1] + radius])
    min_x = min(xs) if xs else -1.0
    max_x = max(xs) if xs else 1.0
    min_y = min(ys) if ys else -1.0
    max_y = max(ys) if ys else 1.0
    span = max(max_x - min_x, max_y - min_y, 1.0)
    padding = 0.18 * span
    center_x = (min_x + max_x) / 2.0
    center_y = (min_y + max_y) / 2.0
    half_span = span / 2.0 + padding
    return (center_x - half_span, center_x + half_span, center_y - half_span, center_y + half_span)


def draw_visual(ax: Any, case: ValidationCase, *, show_point_labels: bool = False) -> None:
    """Draw one validation case from generated visual metadata."""
    visual = case.visual
    labeled_points = visual_points(visual)
    points = [point for _, point in labeled_points]
    simplices = visual_simplices(visual)
    highlighted_simplices = visual.highlighted_simplices
    highlighted_edges = visual.highlighted_edges
    invalid_points = visual.invalid_points
    isolated_points = visual.isolated_points
    duplicate_simplices = list(visual.duplicate_simplices)
    circle = visual_circle(visual)

    palette = ["#7dd3fc", "#fca5a5", "#86efac", "#fde68a", "#c4b5fd"]
    for simplex_index, simplex in enumerate(simplices):
        polygon_points = simplex_points(points, simplex, f"visual.simplices[{simplex_index}]")
        facecolor = "#fb7185" if simplex_index in highlighted_simplices else palette[simplex_index % len(palette)]
        edgecolor = "#be123c" if simplex_index in highlighted_simplices else "#334155"
        polygon = Polygon(
            polygon_points,
            closed=True,
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=1.6,
            alpha=0.34 if simplex_index in highlighted_simplices else 0.24,
        )
        ax.add_patch(polygon)

    for duplicate_index, duplicate in enumerate(duplicate_simplices):
        duplicate_points = simplex_points(points, duplicate, f"visual.duplicate_simplices[{duplicate_index}]")
        polygon = Polygon(
            duplicate_points,
            closed=True,
            facecolor="none",
            edgecolor="#7f1d1d",
            linewidth=2.0,
            hatch="///",
            alpha=0.8,
        )
        ax.add_patch(polygon)
    for edge_index, (left, right) in enumerate(highlighted_edges):
        left_point = point_by_index(points, left, f"visual.highlighted_edges[{edge_index}][0]")
        right_point = point_by_index(points, right, f"visual.highlighted_edges[{edge_index}][1]")
        ax.plot([left_point[0], right_point[0]], [left_point[1], right_point[1]], color="#111827", linewidth=2.4)

    if circle is not None:
        center, radius = circle
        ax.add_patch(Circle(center, radius, fill=False, linestyle="--", linewidth=1.6, edgecolor="#f97316", alpha=0.9))
        ax.scatter([center[0]], [center[1]], s=18, color="#f97316", zorder=5)

    for index, (label, point) in enumerate(labeled_points):
        marker = "x" if index in invalid_points else "o"
        color = "#dc2626" if index in invalid_points else "#0f172a"
        size = 58 if index in invalid_points else 42
        ax.scatter([point[0]], [point[1]], marker=marker, s=size, color=color, zorder=6)
        if index in isolated_points:
            ax.scatter([point[0]], [point[1]], marker="o", s=180, facecolor="none", edgecolor="#dc2626", linewidth=1.8, zorder=5)
        if show_point_labels:
            ax.text(point[0] + 0.03, point[1] + 0.03, label, fontsize=9, weight="bold", color=color)

    limits = axes_limits(points, circle)
    ax.set_xlim(limits[0], limits[1])
    ax.set_ylim(limits[2], limits[3])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_facecolor("#f8fafc")


VALIDATION_HIERARCHY_LAYERS = (
    ValidationHierarchyLayer(
        level=1,
        name="Element Validity",
        scope=("Vertex / Simplex",),
        questions=(
            "Are vertex coordinates finite and valid?",
            "Are element UUIDs present and non-nil?",
            "Do simplices have D+1 distinct vertex keys?",
            "Do simplex vertex keys resolve to distinct coordinates?",
            "Are local neighbor slots shaped and assigned?",
        ),
        color="#dbeafe",
    ),
    ValidationHierarchyLayer(
        level=2,
        name="Combinatorial Consistency",
        scope=("Triangulation Data Structure",),
        questions=(
            "Are UUID-key maps unique and bidirectional?",
            "Does every vertex, simplex, and neighbor reference resolve in the TDS?",
            "Are incident-simplex hints internally consistent?",
            "Are vertex-to-simplex incidence indexes exact?",
            "Are duplicate simplices absent?",
            "Are facets shared by at most two simplices?",
            "Do neighbor links and coherent combinatorial orientation agree?",
        ),
        color="#e0f2fe",
    ),
    ValidationHierarchyLayer(
        level=3,
        name="Intrinsic PL Topology",
        scope=("Triangulation", "Pseudomanifold / PL-Manifold"),
        questions=(
            "Is the simplex neighbor graph connected?",
            "Is every vertex incident to a simplex?",
            "Do facet degrees satisfy the topology guarantee?",
            "Is the true boundary closed?",
            "Do fast ridge-link checks pass when required?",
            "Do complete vertex-link checks pass when required?",
            "Does Euler characteristic match the declared topology?",
            "Is the intrinsic PL manifold orientable?",
        ),
        color="#dcfce7",
    ),
    ValidationHierarchyLayer(
        level=4,
        name="Valid Realization",
        scope=("Euclidean affine-chart", "Toroidal periodic chart", "Spherical realization"),
        questions=(
            "Are affine-chart maximal simplices positively oriented?",
            "Are realized simplices nondegenerate?",
            "Do simplices intersect only along shared faces?",
            "Do toroidal lifts fit valid periodic charts?",
            "Do spherical simplices separate the sphere center?",
            "Does the active realization model support the check?",
        ),
        color="#fef3c7",
    ),
    ValidationHierarchyLayer(
        level=5,
        name="Geometric Predicate Satisfaction",
        scope=("Delaunay Optimality",),
        questions=(
            "Do selected geometric predicates accept every cell?",
            "Do local k=2/k=3 flip predicates and inverses pass?",
            "Do Euclidean/toroidal empty-circumsphere checks pass?",
            "Do spherical empty-cap / ambient hull-facet checks pass?",
            "Is Delaunay optimality certified for the active model?",
        ),
        color="#fee2e2",
    ),
)


def render_validation_hierarchy_figure(output_path: Path) -> None:
    """Render the five-level validation hierarchy overview PNG."""
    figure, axis = plt.subplots(figsize=(13.2, 8.8), facecolor="white", layout="constrained")
    axis.set_axis_off()
    axis.set_xlim(0.0, 1.0)
    axis.set_ylim(0.0, 1.0)

    box_width = 0.86
    box_left = 0.07
    box_top = 0.86
    box_bottom = 0.055
    box_gap = 0.018
    box_inner_gap = 0.042
    level_font_size = 11.8
    name_font_size = 12.8
    scope_font_size = 9.1
    question_font_size = 8.7
    scope_gap = 0.031
    scope_step = 0.024
    question_step_cap = 0.024
    layer_weights = [len(layer.questions) + 2.0 for layer in VALIDATION_HIERARCHY_LAYERS]
    usable_height = box_top - box_bottom - box_gap * (len(VALIDATION_HIERARCHY_LAYERS) - 1)
    box_heights = [usable_height * weight / sum(layer_weights) for weight in layer_weights]

    current_top = box_top
    for index, (layer, box_height) in enumerate(zip(VALIDATION_HIERARCHY_LAYERS, box_heights, strict=True)):
        y = current_top - box_height
        row_center = y + box_height / 2.0
        rectangle = plt.Rectangle(
            (box_left, y),
            box_width,
            box_height,
            facecolor=layer.color,
            edgecolor="#334155",
            linewidth=1.2,
        )
        axis.add_patch(rectangle)
        scope_group_height = scope_gap + scope_step * max(len(layer.scope) - 1, 0)
        label_y = row_center + scope_group_height / 2.0
        axis.text(box_left + 0.035, label_y, f"Level {layer.level}", fontsize=level_font_size, weight="bold", color="#0f172a", wrap=False)
        axis.text(box_left + 0.18, label_y, layer.name, fontsize=name_font_size, weight="bold", color="#0f172a", wrap=False)
        for scope_index, scope in enumerate(layer.scope):
            axis.text(
                box_left + 0.18,
                label_y - scope_gap - scope_step * scope_index,
                scope,
                fontsize=scope_font_size,
                color="#64748b",
                wrap=False,
            )
        question_step = min(question_step_cap, (box_height - box_inner_gap) / max(len(layer.questions) - 1, 1))
        question_top = row_center + question_step * (len(layer.questions) - 1) / 2.0
        for question_index, question in enumerate(layer.questions):
            axis.text(
                box_left + 0.48,
                question_top - question_step * question_index,
                f"- {question}",
                fontsize=question_font_size,
                color="#475569",
                wrap=False,
            )
        if index < len(VALIDATION_HIERARCHY_LAYERS) - 1:
            arrow_x = box_left + box_width / 2.0
            axis.annotate(
                "",
                xy=(arrow_x, y - box_gap * 0.86),
                xytext=(arrow_x, y - box_gap * 0.14),
                arrowprops={"arrowstyle": "->", "linewidth": 1.1, "color": "#64748b"},
            )
        current_top = y - box_gap

    axis.text(0.07, 0.965, "delaunay validation architecture", fontsize=16.5, weight="bold", color="#0f172a", wrap=False)
    axis.text(
        0.07,
        0.928,
        "Level 1 checks local objects; Levels 2-3 are intrinsic; Level 4 checks valid realization; Level 5 checks geometric predicates.",
        fontsize=11.5,
        color="#475569",
        wrap=False,
    )
    save_figure_png(figure, output_path, dpi=180)
    plt.show()
    plt.close(figure)


def wrapped_question(question: str, width: int = 38) -> str:
    """Wrap one validation-family question without adding a text dependency."""
    words = question.split()
    lines: list[str] = []
    line: list[str] = []
    for word in words:
        candidate = " ".join((*line, word))
        if line and len(candidate) > width:
            lines.append(" ".join(line))
            line = [word]
        else:
            line.append(word)
    if line:
        lines.append(" ".join(line))
    return "\n".join(lines)


def glyph_nodes(axis: Any, points: tuple[Point2, ...], *, highlight: tuple[int, ...] = ()) -> None:
    """Draw shared graph nodes with optional invalid-node rings."""
    axis.scatter([point[0] for point in points], [point[1] for point in points], s=28, color="#0f172a", zorder=5)
    for index in highlight:
        point = points[index]
        axis.scatter([point[0]], [point[1]], s=95, facecolor="none", edgecolor="#dc2626", linewidth=1.7, zorder=6)


def glyph_triangle(axis: Any, points: tuple[Point2, Point2, Point2], *, color: str = "#64748b", alpha: float = 0.12) -> None:
    """Draw one filled triangular simplex glyph."""
    axis.add_patch(Polygon(points, closed=True, facecolor=color, edgecolor=color, linewidth=1.5, alpha=alpha))
    glyph_nodes(axis, points)


def glyph_cross(axis: Any, point: Point2) -> None:
    """Mark one failed relation without relying on a text glyph."""
    radius = 0.045
    axis.plot([point[0] - radius, point[0] + radius], [point[1] - radius, point[1] + radius], color="#dc2626", linewidth=2.0)
    axis.plot([point[0] - radius, point[0] + radius], [point[1] + radius, point[1] - radius], color="#dc2626", linewidth=2.0)


def draw_level_1_glyph(axis: Any, family_index: int) -> None:
    """Draw distinct local element-validity failure witnesses."""
    if family_index == 0:
        axis.scatter([0.50], [0.50], s=38, color="#0f172a")
        axis.text(0.50, 0.28, "coords = [NaN, 0]", ha="center", fontsize=8.5, color="#991b1b")
        glyph_cross(axis, (0.50, 0.50))
    elif family_index == 1:
        axis.add_patch(plt.Rectangle((0.18, 0.34), 0.64, 0.34, facecolor="white", edgecolor="#64748b", linewidth=1.4))
        axis.text(0.28, 0.55, "UUID", fontsize=9, weight="bold", color="#475569")
        axis.text(0.58, 0.55, "nil", fontsize=10, weight="bold", color="#991b1b")
        glyph_cross(axis, (0.73, 0.55))
    elif family_index == 2:
        points = ((0.22, 0.28), (0.78, 0.28), (0.50, 0.72))
        glyph_triangle(axis, points)
        for label, point in zip(("A", "A", "B"), points, strict=True):
            axis.text(point[0], point[1] + 0.09, label, ha="center", fontsize=9, weight="bold")
        glyph_nodes(axis, points, highlight=(0, 1))
    elif family_index == 3:
        axis.text(0.20, 0.66, "key A", fontsize=8.5, ha="center")
        axis.text(0.20, 0.34, "key B", fontsize=8.5, ha="center")
        axis.annotate("", xy=(0.72, 0.50), xytext=(0.32, 0.66), arrowprops={"arrowstyle": "->", "color": "#64748b"})
        axis.annotate("", xy=(0.72, 0.50), xytext=(0.32, 0.34), arrowprops={"arrowstyle": "->", "color": "#64748b"})
        axis.scatter([0.72], [0.50], s=38, color="#0f172a")
        glyph_cross(axis, (0.72, 0.50))
    else:
        points = ((0.20, 0.26), (0.80, 0.26), (0.50, 0.74))
        glyph_triangle(axis, points)
        for midpoint in ((0.50, 0.26), (0.35, 0.50)):
            axis.add_patch(plt.Rectangle((midpoint[0] - 0.035, midpoint[1] - 0.035), 0.07, 0.07, facecolor="#86efac", edgecolor="#166534"))
        missing = (0.65, 0.50)
        axis.add_patch(plt.Rectangle((missing[0] - 0.035, missing[1] - 0.035), 0.07, 0.07, facecolor="white", edgecolor="#dc2626", linestyle="--"))


def draw_level_2_glyph(axis: Any, family_index: int) -> None:
    """Draw distinct combinatorial-consistency failure witnesses."""
    if family_index == 0:
        axis.text(0.22, 0.52, "UUID", fontsize=9, ha="center")
        axis.text(0.78, 0.52, "key", fontsize=9, ha="center")
        axis.annotate("", xy=(0.68, 0.58), xytext=(0.32, 0.58), arrowprops={"arrowstyle": "->", "color": "#64748b"})
        axis.annotate("", xy=(0.32, 0.42), xytext=(0.68, 0.42), arrowprops={"arrowstyle": "->", "color": "#dc2626", "linestyle": "--"})
        glyph_cross(axis, (0.50, 0.42))
    elif family_index == 1:
        axis.text(0.22, 0.52, "simplex", fontsize=8.5, ha="center")
        axis.annotate("", xy=(0.72, 0.52), xytext=(0.34, 0.52), arrowprops={"arrowstyle": "->", "color": "#64748b"})
        axis.add_patch(Circle((0.76, 0.52), 0.08, fill=False, edgecolor="#dc2626", linestyle="--"))
        axis.text(0.76, 0.34, "missing ref", fontsize=8, ha="center", color="#991b1b")
    elif family_index == 2:
        points = ((0.14, 0.24), (0.52, 0.24), (0.33, 0.62))
        glyph_triangle(axis, points)
        vertex = (0.80, 0.66)
        axis.scatter([vertex[0]], [vertex[1]], s=32, color="#0f172a")
        axis.annotate("hint", xy=(0.33, 0.45), xytext=vertex, fontsize=8, color="#dc2626", arrowprops={"arrowstyle": "->", "color": "#dc2626"})
        glyph_cross(axis, (0.57, 0.55))
    elif family_index == 3:
        vertex = (0.18, 0.50)
        simplices = ((0.60, 0.68), (0.60, 0.32))
        axis.scatter([vertex[0]], [vertex[1]], s=32, color="#0f172a")
        for simplex in simplices:
            axis.plot([vertex[0], simplex[0]], [vertex[1], simplex[1]], color="#64748b", linewidth=1.3)
            axis.scatter([simplex[0]], [simplex[1]], marker="s", s=38, color="#0f172a")
        axis.text(0.78, 0.50, "index: [top]", fontsize=8, ha="center")
        glyph_cross(axis, simplices[1])
    elif family_index == 4:
        points = ((0.22, 0.28), (0.78, 0.28), (0.50, 0.72))
        glyph_triangle(axis, points, color="#7dd3fc", alpha=0.28)
        shifted = tuple((x + 0.035, y + 0.025) for x, y in points)
        axis.add_patch(Polygon(shifted, closed=True, facecolor="#fca5a5", edgecolor="#991b1b", linewidth=1.5, alpha=0.30))
        axis.text(0.46, 0.48, "s1", fontsize=8, weight="bold")
        axis.text(0.56, 0.43, "s2", fontsize=8, weight="bold", color="#991b1b")
    elif family_index == 5:
        shared = ((0.42, 0.30), (0.42, 0.70))
        for apex in ((0.12, 0.50), (0.72, 0.78), (0.78, 0.22)):
            glyph_triangle(axis, (shared[0], shared[1], apex), color="#fca5a5", alpha=0.20)
        axis.plot([0.42, 0.42], [0.30, 0.70], color="#dc2626", linewidth=3.0)
        axis.text(0.53, 0.50, "degree 3", fontsize=8.5, color="#991b1b")
    else:
        left = ((0.12, 0.28), (0.50, 0.28), (0.31, 0.66))
        right = ((0.50, 0.28), (0.88, 0.28), (0.69, 0.66))
        glyph_triangle(axis, left, alpha=0.16)
        glyph_triangle(axis, right, alpha=0.16)
        axis.annotate("", xy=(0.58, 0.50), xytext=(0.42, 0.50), arrowprops={"arrowstyle": "->", "color": "#dc2626", "linewidth": 1.7})
        glyph_cross(axis, (0.50, 0.50))


def draw_euler_characteristic_glyph(axis: Any) -> None:
    """Draw a planar simplicial complex whose Euler characteristic is not two."""
    left = ((0.14, 0.39), (0.42, 0.39), (0.42, 0.73))
    right = ((0.42, 0.73), (0.70, 0.73), (0.86, 0.41))
    glyph_triangle(axis, left, color="#7dd3fc", alpha=0.24)
    glyph_triangle(axis, right, color="#86efac", alpha=0.24)
    axis.text(0.50, 0.275, "V - E + F = 1 ≠ 2", fontsize=9, weight="bold", ha="center", color="#991b1b")


def draw_mobius_glyph(axis: Any) -> None:
    """Draw a projected Möbius strip witnessing intrinsic non-orientability."""
    samples = 24
    centerline: list[Point2] = []
    upper: list[Point2] = []
    lower: list[Point2] = []
    for sample in range(samples + 1):
        angle = 2.0 * math.pi * sample / samples
        projected_edges: list[Point2] = []
        for offset in (-0.28, 0.28):
            radius = 1.0 + offset * math.cos(angle / 2.0)
            x = radius * math.cos(angle)
            y = 0.58 * radius * math.sin(angle) + 0.34 * offset * math.sin(angle / 2.0)
            projected_edges.append((0.50 + 0.29 * x, 0.51 + 0.29 * y))
        lower.append(projected_edges[0])
        upper.append(projected_edges[1])
        centerline.append(((projected_edges[0][0] + projected_edges[1][0]) / 2.0, (projected_edges[0][1] + projected_edges[1][1]) / 2.0))
    for sample in range(samples):
        face = (lower[sample], lower[sample + 1], upper[sample + 1], upper[sample])
        shade = "#7dd3fc" if sample < samples // 2 else "#c4b5fd"
        axis.add_patch(Polygon(face, closed=True, facecolor=shade, edgecolor="#64748b", linewidth=0.45, alpha=0.52))
    axis.plot([point[0] for point in centerline], [point[1] for point in centerline], color="#334155", linewidth=1.0, linestyle="--")
    top = centerline[6]
    bottom = centerline[18]
    axis.annotate("", xy=(top[0], top[1] + 0.13), xytext=top, arrowprops={"arrowstyle": "->", "color": "#dc2626", "linewidth": 1.8})
    axis.annotate("", xy=(bottom[0], bottom[1] - 0.06), xytext=bottom, arrowprops={"arrowstyle": "->", "color": "#dc2626", "linewidth": 1.8})
    axis.text(0.50, 0.195, "transported normal returns reversed", fontsize=8.2, ha="center", color="#991b1b")


def draw_level_3_glyph(axis: Any, family_index: int) -> None:  # noqa: C901
    """Draw distinct intrinsic-topology failure witnesses."""
    if family_index == 0:
        for center in ((0.25, 0.50), (0.75, 0.50)):
            nodes = ((center[0] - 0.10, 0.40), (center[0], 0.64), (center[0] + 0.10, 0.40))
            glyph_triangle(axis, nodes, alpha=0.14)
        glyph_cross(axis, (0.50, 0.50))
    elif family_index == 1:
        points = ((0.14, 0.28), (0.56, 0.28), (0.35, 0.66))
        glyph_triangle(axis, points, alpha=0.16)
        isolated = (0.82, 0.52)
        axis.scatter([isolated[0]], [isolated[1]], s=32, color="#0f172a")
        axis.add_patch(Circle(isolated, 0.10, fill=False, edgecolor="#dc2626", linestyle="--", linewidth=1.7))
        axis.text(0.82, 0.33, "star = empty", fontsize=8, ha="center", color="#991b1b")
    elif family_index == 2:
        axis.add_patch(Circle((0.50, 0.50), 0.28, fill=False, edgecolor="#64748b", linewidth=1.6))
        axis.plot([0.33, 0.67], [0.72, 0.72], color="#dc2626", linewidth=3.0)
        axis.text(0.50, 0.32, "closed model", fontsize=8, ha="center")
        axis.text(0.50, 0.82, "degree 1", fontsize=8, ha="center", color="#991b1b")
    elif family_index == 3:
        points = ((0.18, 0.32), (0.38, 0.68), (0.60, 0.32), (0.82, 0.60))
        axis.plot([point[0] for point in points], [point[1] for point in points], color="#64748b", linewidth=2.0)
        glyph_nodes(axis, points, highlight=(0, 3))
        axis.text(0.50, 0.18, "open boundary chain", fontsize=8.5, ha="center", color="#991b1b")
    elif family_index == 4:
        center = (0.50, 0.50)
        ring = ((0.50, 0.78), (0.76, 0.56), (0.65, 0.24), (0.35, 0.24), (0.24, 0.56))
        for point in ring:
            axis.plot([center[0], point[0]], [center[1], point[1]], color="#cbd5e1", linewidth=1.0)
        axis.plot([point[0] for point in ring[:-1]], [point[1] for point in ring[:-1]], color="#64748b", linewidth=1.7)
        glyph_nodes(axis, ring)
        glyph_cross(axis, (0.37, 0.67))
    elif family_index == 5:
        center = (0.50, 0.50)
        left = ((0.16, 0.38), (0.28, 0.70), (0.40, 0.38))
        right = ((0.60, 0.62), (0.72, 0.30), (0.84, 0.62))
        for component in (left, right):
            axis.plot([point[0] for point in (*component, component[0])], [point[1] for point in (*component, component[0])], color="#64748b", linewidth=1.5)
        axis.scatter([center[0]], [center[1]], s=36, color="#dc2626")
        axis.text(0.50, 0.82, "disconnected link", fontsize=8.5, ha="center", color="#991b1b")
    elif family_index == 6:
        draw_euler_characteristic_glyph(axis)
    else:
        draw_mobius_glyph(axis)


def draw_level_4_glyph(axis: Any, family_index: int) -> None:
    """Draw distinct realization-validity failure witnesses."""
    if family_index == 0:
        points = ((0.22, 0.28), (0.78, 0.28), (0.50, 0.72))
        glyph_triangle(axis, points, color="#fde68a", alpha=0.30)
        axis.annotate("", xy=(0.35, 0.31), xytext=(0.62, 0.31), arrowprops={"arrowstyle": "->", "color": "#dc2626", "connectionstyle": "arc3,rad=-0.35"})
    elif family_index == 1:
        points = ((0.18, 0.50), (0.50, 0.50), (0.82, 0.50))
        axis.plot([0.18, 0.82], [0.50, 0.50], color="#dc2626", linewidth=2.5)
        glyph_nodes(axis, points)
        axis.text(0.50, 0.28, "zero area", fontsize=8.5, ha="center", color="#991b1b")
    elif family_index == 2:
        vertices = ((0.20, 0.72), (0.80, 0.72), (0.20, 0.22), (0.80, 0.22))
        lower_simplex = (vertices[0], vertices[2], vertices[3])
        upper_simplex = (vertices[0], vertices[1], vertices[2])
        glyph_triangle(axis, lower_simplex, color="#38bdf8", alpha=0.34)
        glyph_triangle(axis, upper_simplex, color="#fb7185", alpha=0.34)
        glyph_nodes(axis, vertices)
        axis.scatter([0.46], [0.50], marker="x", s=80, color="#dc2626", linewidth=2.2, zorder=6)
        axis.text(0.50, 0.86, "overlapping interiors", fontsize=8.5, ha="center", color="#991b1b")
    elif family_index == 3:
        axis.add_patch(plt.Rectangle((0.18, 0.22), 0.64, 0.56, facecolor="none", edgecolor="#64748b", linewidth=1.5))
        axis.scatter([0.24, 0.76], [0.42, 0.58], s=30, color="#0f172a")
        axis.annotate("", xy=(0.86, 0.58), xytext=(0.76, 0.58), arrowprops={"arrowstyle": "->", "color": "#dc2626"})
        axis.scatter([0.14], [0.58], s=55, facecolor="none", edgecolor="#dc2626", linestyle="--")
        axis.text(0.50, 0.84, "lift mismatch", fontsize=8.5, ha="center", color="#991b1b")
    elif family_index == 4:
        axis.add_patch(Circle((0.50, 0.50), 0.30, fill=False, edgecolor="#64748b", linewidth=1.5))
        axis.plot([0.26, 0.76], [0.66, 0.62], color="#dc2626", linewidth=2.0)
        axis.scatter([0.50], [0.50], s=30, color="#0f172a")
        axis.text(0.50, 0.39, "center", fontsize=8, ha="center")
        axis.text(0.50, 0.82, "same side", fontsize=8, ha="center", color="#991b1b")
    else:
        axis.add_patch(plt.Rectangle((0.18, 0.36), 0.64, 0.30, facecolor="white", edgecolor="#64748b", linewidth=1.4))
        axis.text(0.36, 0.52, "model", fontsize=9, weight="bold", ha="center")
        axis.text(0.66, 0.52, "unsupported", fontsize=8.5, ha="center", color="#991b1b")
        glyph_cross(axis, (0.78, 0.52))


def draw_validation_family_glyph(axis: Any, level: int, family_index: int) -> None:
    """Dispatch to a distinct witness for every Level 1-4 validation family."""
    if level == 1:
        draw_level_1_glyph(axis, family_index)
    elif level == 2:
        draw_level_2_glyph(axis, family_index)
    elif level == 3:
        draw_level_3_glyph(axis, family_index)
    elif level == 4:
        draw_level_4_glyph(axis, family_index)
    else:
        raise ValueError(f"layer maps support Levels 1-4, got {level}")
    axis.text(0.12, 0.86, f"L{level}.{family_index + 1}", fontsize=8.5, weight="bold", color="#475569")
    axis.set_xlim(0.0, 1.0)
    axis.set_ylim(0.0, 1.0)
    axis.set_axis_off()


def render_validation_layer_map(case: ValidationCase, output_path: Path) -> None:
    """Render every implemented validation family owned by one layer."""
    layer = VALIDATION_HIERARCHY_LAYERS[case.level - 1]
    columns = 3
    rows = math.ceil(len(layer.questions) / columns)
    figure, axes = plt.subplots(rows, columns, figsize=(10.8, 2.8 * rows), facecolor="white", layout="constrained")
    flat_axes = list(axes.flat) if hasattr(axes, "flat") else [axes]
    for family_index, (axis, question) in enumerate(zip(flat_axes, layer.questions, strict=False)):
        axis.set_facecolor(layer.color)
        draw_validation_family_glyph(axis, layer.level, family_index)
        axis.text(0.50, 0.06, wrapped_question(question), fontsize=9.2, color="#0f172a", ha="center", va="bottom")
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_color("#cbd5e1")
            spine.set_linewidth(1.0)
    for axis in flat_axes[len(layer.questions) :]:
        axis.set_visible(False)
    figure.suptitle(f"Level {layer.level} — {layer.name}", fontsize=16, weight="bold", color="#0f172a")
    save_figure_png(figure, output_path, dpi=220)
    plt.show()
    plt.close(figure)


def render_validation_case_figure(case: ValidationCase, output_path: Path) -> None:
    """Render a complete layer map, retaining the focused Level 5 witness."""
    if case.level < 5:
        render_validation_layer_map(case, output_path)
        return
    figure, axis = plt.subplots(figsize=(4.8, 4.7), facecolor="white", layout="constrained")
    draw_visual(axis, case, show_point_labels=False)
    figure.supxlabel(
        wrapped_question(VALIDATION_HIERARCHY_LAYERS[4].questions[2]),
        fontsize=11.0,
        color="#0f172a",
    )
    save_figure_png(figure, output_path, dpi=240)
    plt.show()
    plt.close(figure)


clear_stale_validation_figures()
render_validation_hierarchy_figure(HIERARCHY_FIGURE_PATH)
print(f"validation hierarchy figure: {HIERARCHY_FIGURE_PATH}")
for case_index, case in enumerate(validation_cases):
    validate_case_visual_invariants(case, case_index)

for case in validation_cases:
    output_path = validation_level_figure_path(case)
    render_validation_case_figure(case, output_path)
    print(f"Level {case.level} figure: {output_path}")


## 5. Copyable artifact summary

In [ ]:
"""Print the generated validation-demo artifact paths."""


if not DEMO_PATH.is_file():
    raise FileNotFoundError(f"validation demo JSON was not written: {DEMO_PATH}")

missing_figures = [figure_path for figure_path in ALL_VALIDATION_FIGURE_PATHS if not figure_path.is_file()]
if missing_figures:
    raise FileNotFoundError(f"validation figures were not written: {missing_figures}")

print(f"JSON artifact: {DEMO_PATH}")
print(f"Hierarchy: {HIERARCHY_FIGURE_PATH}")
print("Validation level figures:")
for figure_path in VALIDATION_FIGURE_PATHS:
    print(f"- {figure_path}")